In [1]:
import os
import gc
import random
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

from tqdm.auto import tqdm
from sklearn.model_selection import LeaveOneGroupOut, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from tensorflow.keras.layers import (
    Input, Conv1D, BatchNormalization, Activation, Dropout,
    Bidirectional, GRU, MultiHeadAttention, Add, LayerNormalization,
    GlobalAveragePooling1D, GlobalMaxPooling1D, Concatenate, Dense
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)

SEED = 42
BATCH_SIZE = 32
EPOCHS = 50

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

FEATURE_ROOT = Path(r"D:\LOSO_MEDIAPIPE")
FEATURE_DIR = FEATURE_ROOT / "features"
METADATA_PATH = FEATURE_ROOT / "metadata.csv"
MODEL_DIR = FEATURE_ROOT / "loso_models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

assert FEATURE_DIR.exists(), f"Features folder not found: {FEATURE_DIR}"
assert METADATA_PATH.exists(), f"Metadata not found: {METADATA_PATH}"

print("TensorFlow:", tf.__version__)

TensorFlow: 2.19.0


In [2]:
metadata = pd.read_csv(METADATA_PATH)

required_columns = {
    "feature_file", "video_path", "signer", "class_name", "label"
}
assert required_columns.issubset(metadata.columns)

X_list, y_list, groups_list = [], [], []
expected_shape = None

for row in tqdm(metadata.itertuples(index=False),
                total=len(metadata),
                desc="Loading features"):

    feature_path = FEATURE_DIR / row.feature_file

    if not feature_path.exists():
        raise FileNotFoundError(f"Missing feature: {feature_path}")

    with np.load(feature_path) as archive:
        landmarks = archive["landmarks"].astype(np.float32)

    if landmarks.ndim != 2 or landmarks.shape[1] != 225:
        raise ValueError(
            f"Invalid feature shape in {feature_path}: {landmarks.shape}"
        )

    if expected_shape is None:
        expected_shape = landmarks.shape

    if landmarks.shape != expected_shape:
        raise ValueError(
            f"Inconsistent sequence shape in {feature_path}: "
            f"{landmarks.shape}; expected {expected_shape}"
        )

    X_list.append(landmarks)
    y_list.append(int(row.label))
    groups_list.append(row.signer)

X = np.stack(X_list).astype(np.float32)
y_original = np.asarray(y_list, dtype=np.int32)
groups = np.asarray(groups_list)

# Ensure class labels are contiguous: 0, 1, ..., num_classes - 1.
label_values = np.sort(np.unique(y_original))
label_to_index = {label: index for index, label in enumerate(label_values)}
y = np.asarray([label_to_index[label] for label in y_original], dtype=np.int32)

TIME_STEPS = X.shape[1]
NUM_FEATURES = X.shape[2]
NUM_CLASSES = len(label_values)

print("X shape:", X.shape)
print("Time steps:", TIME_STEPS)
print("Features per frame:", NUM_FEATURES)
print("Classes:", NUM_CLASSES)
print("Signers:", np.unique(groups))

Loading features:   0%|          | 0/3200 [00:00<?, ?it/s]

X shape: (3200, 30, 225)
Time steps: 30
Features per frame: 225
Classes: 50
Signers: ['S1' 'S2' 'S3' 'S4']


In [3]:
logo = LeaveOneGroupOut()
folds = list(logo.split(X, y, groups))

print("Total LOSO folds:", len(folds))

for fold_number, (train_idx, test_idx) in enumerate(folds, start=1):
    print(
        f"Fold {fold_number}: "
        f"train={np.unique(groups[train_idx])}, "
        f"test={np.unique(groups[test_idx])}"
    )

Total LOSO folds: 4
Fold 1: train=['S2' 'S3' 'S4'], test=['S1']
Fold 2: train=['S1' 'S3' 'S4'], test=['S2']
Fold 3: train=['S1' 'S2' 'S4'], test=['S3']
Fold 4: train=['S1' 'S2' 'S3'], test=['S4']


In [4]:
def standardize_train_val_test(X_train, X_val, X_test):
    """
    Fit normalization using training signers only.
    Missing landmark values remain zero.
    """
    observed_train = X_train != 0.0

    count = observed_train.sum(axis=(0, 1)).astype(np.float32)
    count[count == 0] = 1.0

    mean = (X_train * observed_train).sum(axis=(0, 1)) / count

    squared_error = ((X_train - mean) ** 2) * observed_train
    std = np.sqrt(squared_error.sum(axis=(0, 1)) / count)
    std[std < 1e-6] = 1.0

    def transform(data):
        observed = data != 0.0
        transformed = (data - mean) / std
        transformed[~observed] = 0.0
        return transformed.astype(np.float32)

    return transform(X_train), transform(X_val), transform(X_test)


def augment_batch(batch_x, noise_std=0.008, landmark_dropout=0.03):
    """
    Add small noise only to detected landmarks.
    Missing landmarks remain zero.
    """
    augmented = batch_x.copy()

    batch_size, frames, _ = augmented.shape
    points = augmented.reshape(batch_size, frames, 75, 3)

    valid_landmarks = np.any(points != 0.0, axis=-1, keepdims=True)

    noise = np.random.normal(
        0.0, noise_std, size=points.shape
    ).astype(np.float32)

    points = points + noise * valid_landmarks

    drop_mask = np.random.random(
        size=(batch_size, frames, 75, 1)
    ) < landmark_dropout

    points = np.where(drop_mask & valid_landmarks, 0.0, points)

    return points.reshape(batch_size, frames, NUM_FEATURES).astype(np.float32)


class LandmarkGenerator(tf.keras.utils.Sequence):
    def __init__(self, X_data, y_data, batch_size=32,
                 augment=False, shuffle=True):
        super().__init__()

        self.X_data = X_data
        self.y_data = y_data
        self.batch_size = batch_size
        self.augment = augment
        self.shuffle = shuffle
        self.indices = np.arange(len(X_data))

        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.indices) / self.batch_size))

    def __getitem__(self, index):
        batch_indices = self.indices[
            index * self.batch_size:(index + 1) * self.batch_size
        ]

        batch_x = self.X_data[batch_indices].copy()
        batch_y = self.y_data[batch_indices]

        if self.augment:
            batch_x = augment_batch(batch_x)

        return batch_x, batch_y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

In [5]:
def build_model():
    inputs = Input(shape=(TIME_STEPS, NUM_FEATURES))

    x = Conv1D(128, kernel_size=3, padding="same", use_bias=False)(inputs)
    x = BatchNormalization()(x)
    x = Activation("gelu")(x)
    x = Dropout(0.20)(x)

    x = Conv1D(128, kernel_size=3, padding="same", use_bias=False)(x)
    x = BatchNormalization()(x)
    x = Activation("gelu")(x)
    x = Dropout(0.20)(x)

    x = Bidirectional(
        GRU(96, return_sequences=True, dropout=0.20)
    )(x)

    attention = MultiHeadAttention(
        num_heads=4,
        key_dim=48,
        dropout=0.15
    )(x, x)

    x = Add()([x, attention])
    x = LayerNormalization()(x)

    avg_pool = GlobalAveragePooling1D()(x)
    max_pool = GlobalMaxPooling1D()(x)
    x = Concatenate()([avg_pool, max_pool])

    x = Dense(192, activation="gelu")(x)
    x = Dropout(0.40)(x)

    outputs = Dense(NUM_CLASSES, activation="softmax")(x)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.AdamW(
            learning_rate=5e-4,
            weight_decay=1e-4
        ),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"]
    )

    return model

model = build_model()
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 30, 225)           │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1d (Conv1D)               │ (None, 30, 128)           │          86,400 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization           │ (None, 30, 128)           │             512 │ conv1d[0][0]               │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation (Activation)       │ (None, 30, 128)           │               0 │ batch_normalization[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout (Dropout)             │ (None, 30, 128)           │               0 │ activation[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1d_1 (Conv1D)             │ (None, 30, 128)           │          49,152 │ dropout[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_1         │ (None, 30, 128)           │             512 │ conv1d_1[0][0]             │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_1 (Activation)     │ (None, 30, 128)           │               0 │ batch_normalization_1[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_1 (Dropout)           │ (None, 30, 128)           │               0 │ activation_1[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ bidirectional (Bidirectional) │ (None, 30, 192)           │         130,176 │ dropout_1[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ multi_head_attention          │ (None, 30, 192)           │         148,224 │ bidirectional[0][0],       │
│ (MultiHeadAttention)          │                           │                 │ bidirectional[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ add (Add)                     │ (None, 30, 192)           │               0 │ bidirectional[0][0],       │
│                               │                           │                 │ multi_head_attention[0][0] │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ layer_normalization           │ (None, 30, 192)           │             384 │ add[0][0]                  │
│ (LayerNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ global_average_pooling1d      │ (None, 192)               │               0 │ layer_normalization[0][0]  │
│ (GlobalAveragePooling1D)      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼───────────────

 Total params: 498,930 (1.90 MB)

 Trainable params: 498,418 (1.90 MB)

 Non-trainable params: 512 (2.00 KB)

In [6]:
results = []

for fold_number, (train_idx, test_idx) in enumerate(folds, start=1):
    test_signer = np.unique(groups[test_idx])[0]

    print("\n" + "=" * 70)
    print(f"Fold {fold_number} — Test signer: {test_signer}")
    print("=" * 70)

    X_train_full = X[train_idx]
    y_train_full = y[train_idx]

    X_test = X[test_idx]
    y_test = y[test_idx]

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full,
        y_train_full,
        test_size=0.15,
        random_state=SEED + fold_number,
        stratify=y_train_full
    )

    X_train, X_val, X_test = standardize_train_val_test(
        X_train, X_val, X_test
    )

    train_generator = LandmarkGenerator(
        X_train, y_train,
        batch_size=BATCH_SIZE,
        augment=True,
        shuffle=True
    )

    val_generator = LandmarkGenerator(
        X_val, y_val,
        batch_size=BATCH_SIZE,
        augment=False,
        shuffle=False
    )

    tf.keras.backend.clear_session()
    model = build_model()

    model_path = MODEL_DIR / (
        f"best_fold_{fold_number}_test_{test_signer}.keras"
    )

    callbacks = [
        EarlyStopping(
            monitor="val_accuracy",
            mode="max",
            patience=18,
            restore_best_weights=True,
            verbose=1
        ),

        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=6,
            min_lr=1e-6,
            verbose=1
        ),

        ModelCheckpoint(
            filepath=str(model_path),
            monitor="val_accuracy",
            mode="max",
            save_best_only=True,
            verbose=1
        )
    ]

    model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1
    )

    probabilities = model.predict(
        X_test,
        batch_size=BATCH_SIZE,
        verbose=0
    )

    y_pred = np.argmax(probabilities, axis=1)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(
        y_test, y_pred, average="macro", zero_division=0
    )
    recall = recall_score(
        y_test, y_pred, average="macro", zero_division=0
    )
    macro_f1 = f1_score(
        y_test, y_pred, average="macro", zero_division=0
    )

    print(f"\nAccuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"Macro F1 : {macro_f1:.4f}")

    results.append({
        "test_signer": test_signer,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "macro_f1": macro_f1
    })

    del X_train_full, y_train_full
    del X_train, X_val, X_test
    del y_train, y_val, y_test, model

    gc.collect()


Fold 1 — Test signer: S1

Epoch 1/50
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step - accuracy: 0.0732 - loss: 4.0196
Epoch 1: val_accuracy improved from None to 0.27222, saving model to D:\LOSO_MEDIAPIPE\loso_models\best_fold_1_test_S1.keras
64/64 ━━━━━━━━━━━━━━━━━━━━ 35s 184ms/step - accuracy: 0.1152 - loss: 3.5709 - val_accuracy: 0.2722 - val_loss: 2.5877 - learning_rate: 5.0000e-04
Epoch 2/50
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step - accuracy: 0.3064 - loss: 2.4891
Epoch 2: val_accuracy improved from 0.27222 to 0.61389, saving model to D:\LOSO_MEDIAPIPE\loso_models\best_fold_1_test_S1.keras
64/64 ━━━━━━━━━━━━━━━━━━━━ 9s 142ms/step - accuracy: 0.3564 - loss: 2.2484 - val_accuracy: 0.6139 - val_loss: 1.3469 - learning_rate: 5.0000e-04
Epoch 3/50
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step - accuracy: 0.4744 - loss: 1.6209
Epoch 3: val_accuracy improved from 0.61389 to 0.82500, saving model to D:\LOSO_MEDIAPIPE\loso_models\best_fold_1_test_S1.keras
64/64 ━━━━━━━━━━━━━━━━━━━━ 9s 142ms/step - 